In this problem, we will use gpt2 as an encoder for sentiment classification. Unlike what's done in class, which turns binary classification into prompt + response, we use the gpt2's last hidden state of the last token as the encoder, and then pass through a simple MLP for classification. 

We will finetune such a model and evaluate it on amazon review data.


In [1]:
# !pip install transformers
!pip install datasets

Import necessary libraries

In [2]:
from transformers import GPT2Model, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset, Dataset
import torch.nn as nn
from tqdm import tqdm 
import torch
import numpy as np
import random


# reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'device={device}')


device=cuda


Load gpt2 model and tokenizer  

In [3]:
model_name = "gpt2"  # this is standard gpt2, but can also use larger gpt2 models

# load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Use the end-of-sequence token as the padding token
tokenizer.pad_token_id = tokenizer.eos_token_id  # Set the padding token ID to match the EOS token ID


# we now load the pretrained gpt2 model
gpt2_model  = GPT2Model.from_pretrained(model_name)

/opt/conda/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Load Amazon review dataset

In [4]:

# Load Amazon review dataset 
dataset = load_dataset("amazon_polarity")

train_dataset = dataset['train'].shuffle(seed=42).select(range(2500))
val_dataset = dataset['test'].shuffle(seed=42).select(range(500))

def tokenize_function(examples):
    return tokenizer(examples["content"], padding="max_length", truncation=True, max_length=128)

# Apply tokenizer to the datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Rename 'label' to 'labels' in the dataset to match huggingface trainer
train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")

print(train_dataset.column_names)

# print a training data example
print(train_dataset[0])

['labels', 'title', 'content', 'input_ids', 'attention_mask']
{'labels': 0, 'title': 'Anyone who likes this better than the Pekinpah is a moron.', 'content': "All the pretty people in this film. Even the Rudy character played by Michael Madsen. This is adapted from a Jim Thompson novel for cryin' out loud! These are supposed to be marginal characters, not fashion models. Though McQueen and McGraw were attractive (but check out McQueen's crummy prison haircut) they were believable in the role. Baldwin and Bassinger seem like movie stars trying to act like hard cases. Action wise, the robbery scene in the Pekinpah version was about 100 times more exciting and suspenseful than anything in this re-make.", 'input_ids': [3237, 262, 2495, 661, 287, 428, 2646, 13, 3412, 262, 37602, 2095, 2826, 416, 3899, 4627, 6248, 13, 770, 318, 16573, 422, 257, 5395, 11654, 5337, 329, 3960, 259, 6, 503, 7812, 0, 2312, 389, 4385, 284, 307, 14461, 3435, 11, 407, 6977, 4981, 13, 7486, 1982, 32466, 290, 11130, 1

We now define a GPT2ForClassification which should be iniitalized with either GPT2Model (gpt2_model ) or GPT2ModelBidirectional in its init function.

It should take input tokens and run gpt2_model, then it takes the last hidden state of the last input token, and pass it through an MLP with 128 hidden dimension and ReLU activation, then maps it to logits for binary classification. To be compatible with huggingface trainer, the forward function should return loss when labels are not None.

```python

class GPT2ForBinaryClassification(nn.Module):
    def __init__(self, gpt2_model):
     ....

    def forward(self, input_ids, attention_mask=None, labels=None):
     ....
      # Return logits and loss (if available)
      return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

```

In [9]:
# Implement class GPT2ForBinaryClassification(nn.Module)
class GPT2ForBinaryClassification(nn.Module):
    def __init__(self, gpt2_model):
        super(GPT2ForBinaryClassification, self).__init__()
        self.gpt2 = gpt2_model
        self.fc = nn.Linear(self.gpt2.config.hidden_size, 128)
        self.relu = nn.ReLU()
        self.classifier = nn.Linear(128, 2)

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.gpt2(input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state[:, -1, :]
        x = self.fc(last_hidden_state)
        x = self.relu(x)
        logits = self.classifier(x)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)
        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}


We now define training_args for use with huggingface trainer.

You want to choose your traing_args so that you achieve good evaluation result. Make sure you achieve more than 90, and potentially 92 evaluation accuracy

In [10]:
# Implement training_args 
# data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    logging_steps=100,
    learning_rate=1e-4,
    weight_decay=0.01,
    seed=seed,
)

Define train and eval function based on huggingface's interface

In [11]:
# Define a custom compute_metrics function
def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": (preds == p.label_ids).astype(float).mean().item()}

def train_and_eval(model,training_args, messages):
    trainer  = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,         # Training dataset
        eval_dataset=val_dataset,            # Evaluation dataset
        tokenizer=tokenizer,                 # Tokenizer used
        compute_metrics=compute_metrics,     # Function to compute metrics
    )
    print(f'===== training {messages} =====')
    trainer.train()
    print(f'===== evaluating {messages} =====')
    eval_results =trainer.evaluate()
    print("Evaluation Results:", eval_results)




We now train and evaluate gpt2 classification model

In [12]:

model  = GPT2ForBinaryClassification(gpt2_model)

train_and_eval(model, training_args,"gpt2 classification model")

===== training gpt2 classification model =====


Step,Training Loss
100,0.459100
200,0.204800
300,0.115900


===== evaluating gpt2 classification model =====


Evaluation Results: {'eval_loss': 0.37541499733924866, 'eval_accuracy': 0.92, 'eval_runtime': 0.9868, 'eval_samples_per_second': 506.707, 'eval_steps_per_second': 63.845, 'epoch': 5.0}
